In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
data_path="/content/drive/MyDrive/PetImages"

In [4]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import os

from tensorflow.keras import layers as layer,models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D,MaxPooling2D,Dense,Dropout,Flatten
from tensorflow.keras.optimizers import Adam

In [5]:
IMG_SIZE=224
BATCH_SIZE=32
train_dataset=tf.keras.utils.image_dataset_from_directory(
    data_path,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=(IMG_SIZE,IMG_SIZE),
    batch_size=BATCH_SIZE
)
validation_dataset=tf.keras.utils.image_dataset_from_directory(
    data_path,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(IMG_SIZE,IMG_SIZE),
    batch_size=BATCH_SIZE
)
print(train_dataset.class_names)

Found 11999 files belonging to 2 classes.
Using 9600 files for training.
Found 11999 files belonging to 2 classes.
Using 2399 files for validation.
['Cat', 'Dog']


In [6]:
cnn=models.Sequential([
    layer.Input(shape=(IMG_SIZE,IMG_SIZE,3)),
    layer.Rescaling(1./255),
    layer.Conv2D(32,kernel_size=(3,3),activation='relu'),
    layer.MaxPooling2D((2,2)),
    layer.Conv2D(64,kernel_size=(3,3),activation='relu'),
    layer.MaxPooling2D((2,2)),
    layer.Flatten(),
    layer.Dense(128,activation='relu'),
    layer.Dropout(0.2),
    layer.Dense(2,activation='sigmoid')
])

In [7]:
cnn.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)
cnn.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 186624)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    23,888,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,907,650 (91.20 MB)

 Trainable params: 23,907,650 (91.20 MB)

 Non-trainable params: 0 (0.00 B)

In [8]:
'''import os
import tensorflow as tf

print("Cleaning up corrupt and unsupported files using TensorFlow decoding...")
num_skipped = 0
for folder in ['Cat', 'Dog']:
    folder_path = os.path.join(data_path, folder)
    if os.path.isdir(folder_path):
        for filename in os.listdir(folder_path):
            file_path = os.path.join(folder_path, filename)
            # Skip hidden files or directories
            if filename.startswith('.') or os.path.isdir(file_path):
                continue
            try:
                # Read file raw bytes
                img_bytes = tf.io.read_file(file_path)
                # Try decoding to ensure TensorFlow accepts the format
                tf.io.decode_image(img_bytes, channels=3, expand_animations=False)
            except Exception as e:
                print(f"Removing invalid file: {file_path} (Reason: {e})")
                try:
                    os.remove(file_path)
                    num_skipped += 1
                except Exception as ex:
                    print(f"Could not remove {file_path}: {ex}")

print(f"Cleanup complete. Removed {num_skipped} files.")
'''


Cleaning up corrupt and unsupported files using TensorFlow decoding...


KeyboardInterrupt: 

In [ ]:
# Re-instantiate datasets after cleanup with 'categorical' label mode
train_dataset = tf.keras.utils.image_dataset_from_directory(
    data_path,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

validation_dataset = tf.keras.utils.image_dataset_from_directory(
    data_path,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

# Re-build and re-compile the model to clear state and match the categorical dataset shape
cnn = models.Sequential([
    layer.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
    layer.Rescaling(1./255),
    layer.Conv2D(32, kernel_size=(3,3), activation='relu'),
    layer.MaxPooling2D((2,2)),
    layer.Conv2D(64, kernel_size=(3,3), activation='relu'),
    layer.MaxPooling2D((2,2)),
    layer.Flatten(),
    layer.Dense(128, activation='relu'),
    layer.Dropout(0.2),
    layer.Dense(2, activation='sigmoid')
])

cnn.compile(
    optimizer='SGD',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history = cnn.fit(train_dataset, validation_data=validation_dataset, epochs=25)

Found 11999 files belonging to 2 classes.
Using 9600 files for training.
Found 11999 files belonging to 2 classes.
Using 2399 files for validation.
Epoch 1/25
291/300 ━━━━━━━━━━━━━━━━━━━━ 55s 6s/step - accuracy: 0.5270 - loss: 0.6930 

In [ ]:
print(cnn.evaluate(validation_dataset))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))

plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')

plt.title('CNN Model Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

In [ ]:
import numpy as np
import tensorflow as tf

def classify_image(image_path, model, class_names=['Cat', 'Dog'], img_size=224):
    """
    Loads a single image, preprocesses it, and uses the trained model to predict its class.
    """
    try:
        # Load and resize the image
        img = tf.keras.utils.load_img(image_path, target_size=(img_size, img_size))
        img_array = tf.keras.utils.img_to_array(img)

        # Add batch dimension: (224, 224, 3) -> (1, 224, 224, 3)
        img_array = np.expand_dims(img_array, axis=0)

        # Make prediction
        predictions = model.predict(img_array)

        # Since we used label_mode='categorical' (2 outputs), get the index of the highest probability
        score = tf.nn.softmax(predictions[0])
        predicted_class_idx = np.argmax(predictions[0])
        predicted_class = class_names[predicted_class_idx]
        confidence = predictions[0][predicted_class_idx] * 100

        print(f"Predicted Class: {predicted_class} ({confidence:.2f}% confidence)")
        return img
    except Exception as e:
        print(f"Error processing image: {e}")
        return None

# Example usage: Replace 'path_to_new_image.jpg' with the actual path of your image
# test_image_path = "/content/drive/MyDrive/PetImages/Cat/1.jpg"
# classify_image(test_image_path, cnn)

In [ ]:
# Replace the string below with the path to the new image you want to classify
my_image_path = "/content/drive/MyDrive/PetImages/Cat/dinesh.jpg"

# Run the classification function on your image
img = classify_image(my_image_path, cnn)

# Optionally, display the image
if img:
    import matplotlib.pyplot as plt
    plt.imshow(img)
    plt.axis('off')
    plt.show()